In [1]:
from netgen.meshing import Mesh as NGMesh, MeshPoint, Element1D, Element0D, Pnt
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw, FieldLines, AddFieldLines

import matplotlib.pylab as plt


In [2]:
def MakeMesh(h_max):
    shape = MoveTo(0,0).RectangleC(20,20) \
        .MoveTo(0,1).RectangleC(5,0.5, "el1").Reverse() \
        .MoveTo(0,-1).RectangleC(5,0.5, "el2").Reverse() \
        .Face()
    shape.edges["el.*"].vertices.hpref=1
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))
    return shape,mesh

def MakeMeshRounded(h_max):
    square = MoveTo(0,0).RectangleC(20,20).Face()
    el1 = MoveTo(0,1).RectangleC(5,0.5).Face()
    el1 += MoveTo(2.5,1).Circle(0.25).Face()
    el1 += MoveTo(-2.5,1).Circle(0.25).Face()
    el1.edges.name="el1"
    el2 = MoveTo(0,-1).RectangleC(5,0.5).Face()
    el2 += MoveTo(2.5,-1).Circle(0.25).Face()
    el2 += MoveTo(-2.5,-1).Circle(0.25).Face()
    el2.edges.name="el2"
    geo = square - el1 - el2
    mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=h_max))
    return geo,mesh

In [3]:
h_max = 0.5
geo, mesh = MakeMeshRounded(h_max)
Draw(geo)
Draw (mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': …

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [4]:
def SolveProblem(mesh, FE_order, eps0):
    fes = H1(mesh, order=FE_order, dirichlet="el.*")
    u = fes.TrialFunction()
    v = fes.TestFunction()

    gfu = GridFunction(fes)
    gfu.Interpolate(mesh.BoundaryCF({"el1":1, "el2":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(eps0*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes.FreeDofs())
    gfu.vec.data -= inv@a.mat * gfu.vec
    return gfu

In [5]:
FE_order = 3
eps0 = 8.854e-12
gfu = SolveProblem(mesh, FE_order, eps0)

In [6]:
ea = { "euler_angles" : [-70,0,-40]} 
Draw (gfu, deformation=True, scale=5, **ea);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': […

In [7]:
fes_flux = (mesh, order=FE_order-1)
D = GridFunction(fes_flux, name="D")
E = GridFunction(fes_flux, name="E")
D.Set(-eps0*grad(gfu))
E.Set(-grad(gfu))

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (3487958126.py, line 1)

In [ ]:
Draw (E, mesh, "Flux", vectors= { "grid_size" : 50});

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [ ]:
Draw (Norm(E), mesh, order=3, deformation=True, min=0, max=3);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [ ]:
Draw (D, mesh, "Flux", vectors= { "grid_size" : 40});

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [ ]:
N = 200
p = [(-10 + 0.1*i, -10 + 0.1*j, 0) for i in range(N) for j in range(N) ] 

fieldlines = E._BuildFieldLines(mesh, p, num_fieldlines=300, length=0.3)



Draw(E, mesh,  "X", draw_vol=True, draw_surf=True, objects=[fieldlines], \
     autoscale=True, min = 0, max = 2, settings={"Objects": {"Surface": False}});

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Surface': False…

In [ ]:
energy = 0.5 * Integrate(eps0 * InnerProduct(E, E), mesh)
print(energy)

9.045553455936279e-11
